# Two-dimensional matrix mapping

Each thread writes one matrix entry. Both kernels use the same formula; they differ in whether neighboring `threadIdx.x` values select columns or rows. This lets us compare the effect of the thread-to-data mapping.

## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell displays the selected device and creates a folder for the CUDA source files.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov 14 06:35:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

The error-checking macro is independent of the timer. `GetCounter()` returns elapsed milliseconds and waits for the recorded work to finish.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}

TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}

void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}

float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## Kernels and host program

The small run prints the CPU reference matrix and both GPU matrices, then compares every GPU value with the corresponding CPU value. The timing run repeats each GPU mapping 100 times and reports the average time per launch. The CPU matrix is used for correctness checking, not for a CPU-versus-GPU speed comparison; only kernel execution is timed.

In [5]:
%%writefile code/chapter02_matrix_write.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>

__global__ void fill_columns(float *matrix, int rows, int cols) {
    const int c = blockIdx.x * blockDim.x + threadIdx.x;
    const int r = blockIdx.y * blockDim.y + threadIdx.y;
    if (r < rows && c < cols) {
        matrix[r * cols + c] = float(r * cols + c);
    }
}

__global__ void fill_rows(float *matrix, int rows, int cols) {
    const int r = blockIdx.x * blockDim.x + threadIdx.x;
    const int c = blockIdx.y * blockDim.y + threadIdx.y;
    if (r < rows && c < cols) {
        matrix[r * cols + c] = float(r * cols + c);
    }
}

void launch(int mapping, float *matrix, int rows, int cols) {
    const dim3 block(32, 8);
    if (mapping == 0) {
        const dim3 grid((cols + block.x - 1) / block.x,
                        (rows + block.y - 1) / block.y);
        fill_columns<<<grid, block>>>(matrix, rows, cols);
    } else {
        const dim3 grid((rows + block.x - 1) / block.x,
                        (cols + block.y - 1) / block.y);
        fill_rows<<<grid, block>>>(matrix, rows, cols);
    }
    CUDA_CHECK(cudaGetLastError());
}

void fill_cpu(float *matrix, int rows, int cols) {
    for (int r = 0; r < rows; ++r) {
        for (int c = 0; c < cols; ++c) {
            matrix[r * cols + c] = float(r * cols + c);
        }
    }
}

void print_matrix(const char *title, const float *matrix,
                  int rows, int cols) {
    std::printf("%s\n", title);
    for (int r = 0; r < rows; ++r) {
        for (int c = 0; c < cols; ++c) {
            std::printf("%6.0f", matrix[r * cols + c]);
        }
        std::printf("\n");
    }
}

void check_and_print(int mapping, float *d_matrix, const float *h_cpu,
                     int rows, int cols, bool print_gpu_matrix) {
    const size_t count = size_t(rows) * size_t(cols);
    const size_t bytes = count * sizeof(float);
    float *h_gpu = (float*)std::malloc(bytes);
    if (h_gpu == nullptr) {
        std::fprintf(stderr, "Host allocation failed.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaMemset(d_matrix, 0xff, bytes));
    launch(mapping, d_matrix, rows, cols);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_gpu, d_matrix, bytes, cudaMemcpyDeviceToHost));
    for (size_t i = 0; i < count; ++i) {
        if (!std::isfinite(h_gpu[i]) || h_gpu[i] != h_cpu[i]) {
            std::fprintf(stderr,
                         "CPU/GPU mismatch at element %zu: CPU=%g, GPU=%g\n",
                         i, h_cpu[i], h_gpu[i]);
            std::exit(EXIT_FAILURE);
        }
    }
    std::printf("CPU vs GPU (%s): PASS, %zu values match\n",
                mapping == 0 ? "x-to-column" : "x-to-row", count);
    if (print_gpu_matrix) {
        print_matrix(mapping == 0 ? "GPU, x-to-column:" : "GPU, x-to-row:",
                     h_gpu, rows, cols);
    }
    std::free(h_gpu);
}

float time_mapping(int mapping, float *d_matrix, int rows, int cols,
                   int repeats, TimingGPU& timer) {
    for (int k = 0; k < 5; ++k) launch(mapping, d_matrix, rows, cols);
    CUDA_CHECK(cudaDeviceSynchronize());
    timer.StartCounter();
    for (int k = 0; k < repeats; ++k) launch(mapping, d_matrix, rows, cols);
    return timer.GetCounter() / repeats;
}

int main(int argc, char **argv) {
    if (argc != 2 || (std::strcmp(argv[1], "--show") != 0 &&
                      std::strcmp(argv[1], "--time") != 0)) {
        std::fprintf(stderr, "Usage: %s --show | --time\n", argv[0]);
        return EXIT_FAILURE;
    }
    CUDA_CHECK(cudaSetDevice(0));
    const bool show = std::strcmp(argv[1], "--show") == 0;
    const int rows = show ? 6 : 2048;
    const int cols = show ? 10 : 2048;
    const size_t count = size_t(rows) * size_t(cols);
    const size_t bytes = count * sizeof(float);

    float *h_cpu = (float*)std::malloc(bytes);
    float *d_matrix = nullptr;
    if (h_cpu == nullptr) {
        std::fprintf(stderr, "Host allocation failed.\n");
        return EXIT_FAILURE;
    }
    fill_cpu(h_cpu, rows, cols);
    CUDA_CHECK(cudaMalloc((void**)&d_matrix, bytes));

    if (show) print_matrix("CPU reference:", h_cpu, rows, cols);
    for (int mapping = 0; mapping < 2; ++mapping) {
        check_and_print(mapping, d_matrix, h_cpu, rows, cols, show);
    }
    if (!show) {
        TimingGPU timer;
        const int repeats = 100;
        float ms[2];
        std::printf("\nGPU kernel timing for a %d x %d matrix (%d launches):\n",
                    rows, cols, repeats);
        for (int mapping = 0; mapping < 2; ++mapping) {
            ms[mapping] = time_mapping(mapping, d_matrix, rows, cols,
                                       repeats, timer);
            std::printf("%s: %.6f ms/launch\n",
                        mapping == 0 ? "x-to-column" : "x-to-row", ms[mapping]);
        }
        std::printf("T(x-to-row) / T(x-to-column) = %.3f\n",
                    ms[1] / ms[0]);
    }

    CUDA_CHECK(cudaFree(d_matrix));
    std::free(h_cpu);
    return EXIT_SUCCESS;
}


Writing code/chapter02_matrix_write.cu


## Compile

The two kernels and the host program are in one `.cu` file. The timer is compiled alongside it; `CUDA_CHECK` remains in its own header.

In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_matrix_write.cu code/TimingGPU.cu -o matrix_write


## Compare CPU and GPU results

The program prints the CPU reference and the matrices produced by both GPU mappings. It also checks every GPU element against the corresponding CPU element.

In [7]:
import subprocess

result = subprocess.run(
    ['./matrix_write', '--show'], capture_output=True, text=True
)
print(result.stdout, end='')
if result.stderr:
    print(result.stderr, end='')
result.check_returncode()


CPU reference:
     0     1     2     3     4     5     6     7     8     9
    10    11    12    13    14    15    16    17    18    19
    20    21    22    23    24    25    26    27    28    29
    30    31    32    33    34    35    36    37    38    39
    40    41    42    43    44    45    46    47    48    49
    50    51    52    53    54    55    56    57    58    59
CPU vs GPU (x-to-column): PASS, 60 values match
GPU, x-to-column:
     0     1     2     3     4     5     6     7     8     9
    10    11    12    13    14    15    16    17    18    19
    20    21    22    23    24    25    26    27    28    29
    30    31    32    33    34    35    36    37    38    39
    40    41    42    43    44    45    46    47    48    49
    50    51    52    53    54    55    56    57    58    59
CPU vs GPU (x-to-row): PASS, 60 values match
GPU, x-to-row:
     0     1     2     3     4     5     6     7     8     9
    10    11    12    13    14    15    16    17    18    19
    2

## Compare the GPU execution times

This run validates both GPU outputs against the CPU reference, then reports the average time per launch for x-to-column and x-to-row mappings. The final ratio is `T(x-to-row) / T(x-to-column)`; it compares the two GPU access patterns. The CPU matrix is not timed here, since this timing isolates kernel execution and excludes data transfers.

In [8]:
result = subprocess.run(
    ['./matrix_write', '--time'], capture_output=True, text=True
)
print(result.stdout, end='')
if result.stderr:
    print(result.stderr, end='')
result.check_returncode()


CPU vs GPU (x-to-column): PASS, 4194304 values match
CPU vs GPU (x-to-row): PASS, 4194304 values match

GPU kernel timing for a 2048 x 2048 matrix (100 launches):
x-to-column: 0.075920 ms/launch
x-to-row: 0.432558 ms/launch
T(x-to-row) / T(x-to-column) = 5.698
